# 💳 Fraud Detection — SMOTE, Logistic Regression & Random Forest

**Oasis Infobyte Data Analytics — Level 2, Task 3**

This notebook covers the full Task 3 checklist: class imbalance, fraud-vs-legitimate amount EDA, time-of-day analysis, stratified splitting, SMOTE, Logistic Regression, Random Forest, Precision/Recall/F1/ROC-AUC, confusion matrices, ROC curves, feature importance, metric trade-offs and scalability.

## 1. Load, clean and inspect

Fraud detection is a heavily imbalanced classification problem. Accuracy is not sufficient because a model can be highly accurate while missing most fraud cases.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from imblearn.over_sampling import SMOTE

RANDOM_STATE=42
DATA_PATH='../data/creditcard.csv'
RESULTS_DIR='../results'
os.makedirs(RESULTS_DIR,exist_ok=True)
df=pd.read_csv(DATA_PATH)
print('Original shape:',df.shape)
print('Missing values:',int(df.isna().sum().sum()))
print('Duplicate rows:',int(df.duplicated().sum()))
df=df.drop_duplicates().reset_index(drop=True)
print('Cleaned shape:',df.shape)
print(df['Class'].value_counts().sort_index())
print(f'Fraud rate: {df.Class.mean()*100:.4f}%')

### Interpretation

The current cleaned run contains **283,726 transactions**, including **473 fraud cases (0.1667%)**. This extreme imbalance is the central modelling challenge.

## 2. Class distribution, transaction amount and time-of-day EDA

In [ ]:
# Class distribution
counts=df['Class'].value_counts().sort_index()
plt.figure(figsize=(7,4.5)); ax=sns.barplot(x=['Legitimate','Fraud'],y=counts.values)
plt.title('Transaction Class Distribution'); plt.xlabel('Transaction Type'); plt.ylabel('Number of Transactions')
for b,v in zip(ax.patches,counts.values): ax.annotate(f'{v:,}',(b.get_x()+b.get_width()/2,v),ha='center',va='bottom')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'class_distribution.png'),dpi=180,bbox_inches='tight'); plt.show()

# Fraud vs legitimate transaction amounts
plt.figure(figsize=(9,5)); sns.histplot(data=df,x='Amount',hue='Class',bins=80,stat='density',common_norm=False,element='step')
plt.xlim(0,df.Amount.quantile(.995)); plt.title('Transaction Amount Distribution: Fraud vs Legitimate'); plt.xlabel('Transaction Amount'); plt.ylabel('Density')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'amount_distribution.png'),dpi=180,bbox_inches='tight'); plt.show()
print(df.groupby('Class')['Amount'].describe().round(2))

# Time-of-day analysis
df['Hour']=((df['Time']%(24*60*60))//3600).astype(int)
hourly=df.groupby('Hour')['Class'].agg(['count','sum']).rename(columns={'sum':'fraud_count'})
hourly['fraud_rate_pct']=hourly.fraud_count/hourly['count']*100
plt.figure(figsize=(10,5)); sns.lineplot(data=hourly,x=hourly.index,y='fraud_rate_pct',marker='o')
plt.title('Fraud Rate by Hour of Day'); plt.xlabel('Hour of Day'); plt.ylabel('Fraud Rate (%)'); plt.xticks(range(24))
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'fraud_rate_by_hour.png'),dpi=180,bbox_inches='tight'); plt.show()
print(hourly.round(4))

### Interpretation

Amount distributions overlap, so transaction amount alone cannot reliably identify fraud. The hourly analysis is descriptive because the dataset covers only two days and the time field is anonymised.

## 3. Stratified split, scaling and SMOTE

In [ ]:
X=df.drop(columns=['Class','Hour']); y=df['Class']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=RANDOM_STATE)
scaler=StandardScaler(); X_train_scaled=scaler.fit_transform(X_train); X_test_scaled=scaler.transform(X_test)
smote=SMOTE(random_state=RANDOM_STATE); X_train_smote,y_train_smote=smote.fit_resample(X_train_scaled,y_train)
print('Train:',X_train.shape,' Test:',X_test.shape)
print('Fraud in train:',int(y_train.sum()),' Fraud in test:',int(y_test.sum()))
print('After SMOTE:',pd.Series(y_train_smote).value_counts().to_dict())

### Interpretation

Stratification ensures fraud appears in both splits. SMOTE is applied only to the training data so the hold-out test set remains representative of the original class imbalance.

## 4. Train and evaluate the two required models

In [ ]:
models={'Logistic Regression':LogisticRegression(max_iter=2000,random_state=RANDOM_STATE),'Random Forest':RandomForestClassifier(n_estimators=200,random_state=RANDOM_STATE,n_jobs=-1)}
predictions={}; probabilities={}; fitted={}
for name,model in models.items():
    model.fit(X_train_smote,y_train_smote); fitted[name]=model
    predictions[name]=model.predict(X_test_scaled); probabilities[name]=model.predict_proba(X_test_scaled)[:,1]

rows=[]
for name in models:
    rows.append({'Model':name,'Precision':precision_score(y_test,predictions[name],zero_division=0),'Recall':recall_score(y_test,predictions[name],zero_division=0),'F1':f1_score(y_test,predictions[name],zero_division=0),'ROC-AUC':roc_auc_score(y_test,probabilities[name])})
metrics=pd.DataFrame(rows); print(metrics.round(4)); metrics.to_csv(os.path.join(RESULTS_DIR,'model_metrics.csv'),index=False)

### Metric interpretation

**Recall** matters when missed fraud is costly because it measures the proportion of actual fraud detected. However, high recall can create many false positives. **Precision** measures how many alerts are actually fraud, and **F1-score** balances Precision and Recall. ROC-AUC measures ranking performance across thresholds. In production, the threshold should be selected from the cost of missed fraud versus false alerts.

In [ ]:
# Confusion matrices
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
for ax,name in zip(axes,models):
    cm=confusion_matrix(y_test,predictions[name]); sns.heatmap(cm,annot=True,fmt=',d',cmap='Blues',cbar=False,ax=ax)
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_xticklabels(['Legitimate','Fraud']); ax.set_yticklabels(['Legitimate','Fraud'],rotation=0)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'confusion_matrices.png'),dpi=180,bbox_inches='tight'); plt.show()

# ROC curves
plt.figure(figsize=(8,6))
for name in models:
    fpr,tpr,_=roc_curve(y_test,probabilities[name]); auc=roc_auc_score(y_test,probabilities[name]); plt.plot(fpr,tpr,label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'--',label='Chance'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('AUC-ROC Comparison'); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'roc_curve.png'),dpi=180,bbox_inches='tight'); plt.show()

# Random Forest feature importance
fi=pd.DataFrame({'Feature':X.columns,'Importance':fitted['Random Forest'].feature_importances_}).sort_values('Importance',ascending=False)
fi.to_csv(os.path.join(RESULTS_DIR,'random_forest_feature_importance.csv'),index=False)
plt.figure(figsize=(9,6)); sns.barplot(data=fi.head(10),x='Importance',y='Feature'); plt.title('Random Forest — Top 10 Feature Importances')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'feature_importance.png'),dpi=180,bbox_inches='tight'); plt.show()
display(fi.head(10))

### Interpretation

Confusion matrices show false negatives (missed fraud) and false positives (legitimate transactions flagged as fraud). The Random Forest feature importances are dominated by anonymised PCA-derived variables, so they indicate predictive contribution rather than a direct business cause.

## 5. Scalability — 1 million transactions per hour

A production system at this volume would need distributed or streaming ingestion, low-latency feature generation, a versioned feature store, scalable model serving, batch retraining, threshold calibration, drift monitoring, latency monitoring, periodic retraining and human review of high-risk alerts. The notebook is an educational prototype, not a production financial decision system.

## 6. Oasis Infobyte Task 3 checklist

✅ Class imbalance and fraud percentage

✅ Fraud vs non-fraud transaction amount EDA

✅ Time-of-day analysis

✅ Accuracy limitation explained

✅ SMOTE imbalance handling

✅ Stratified train/test split

✅ Logistic Regression + Random Forest

✅ Precision, Recall, F1-score and ROC-AUC

✅ Confusion matrices and ROC curve

✅ Feature importance

✅ Recall-versus-Precision trade-off

✅ Scalability discussion